In [3]:
from google.cloud import bigquery
import pandas as pd
import pandas
from decimal import Decimal
import queue, concurrent.futures
import traceback
import time
import threading
from datetime import datetime, timedelta


# Top Revenue


In [4]:
PROJECT_ID = "clean-pilot-456915-t0"
client = bigquery.Client(project= PROJECT_ID)

c:\Users\Tzuying\Project\BigQuery Pipeline\.venv\Lib\site-packages\google\auth\_default.py:113: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


In [5]:
df = client.query("""  
SELECT 
    t.id,
    t.tranid,
    t.trandate,
    t.status,
    t.type,
    tl.item,
    i.fullname,
    i.class,
    i.manufacturer,          
    c.category,             
    tl.quantity,
    tl.quantitybackordered,
    tl.netamount,
    tl.rate,
    tl.costestimate,
    tl.inventorylocation,    
    tl.custcolfree_goods_checkbox  
FROM `clean-pilot-456915-t0.NetSuite.transaction` t
JOIN `clean-pilot-456915-t0.NetSuite.transactionline` tl ON t.id = tl.transaction
JOIN `clean-pilot-456915-t0.NetSuite.customer` c ON t.entity = c.id
LEFT JOIN `clean-pilot-456915-t0.NetSuite.item` i ON tl.item = i.id
WHERE t.trandate <= DATE_SUB(CURRENT_DATE(), INTERVAL 1 DAY)
  AND t.type IN ('SalesOrd', 'RtnAuth')   -- include returns
  AND t.entity NOT IN (604610, 615265)
  AND c.category != 6                      -- exclude wholesale
  AND tl.mainline = FALSE
  AND tl.taxline = FALSE
  AND tl.inventorylocation = 1
  AND tl.itemtype = 'InvtPart'
  AND i.manufacturer != 'D2'
ORDER BY t.trandate DESC
""").to_dataframe()

c:\Users\Tzuying\Project\BigQuery Pipeline\.venv\Lib\site-packages\google\cloud\bigquery\table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


In [7]:

# deduplicate multiple line of item in one Sales Order
so = df[df["type"]=="SalesOrd"]
so = so.groupby(by = ["tranid", "item"], as_index=False).agg(
    quantity = ("quantity", "sum"), netamount = ("netamount", "sum"),
    costestimate = ("costestimate", "sum"), status = ("status","first"),
    trandate = ("trandate", "first"), type = ("type", "first")
)
# Selected the Analysis timeline
six_months_ago = (datetime.today() - timedelta(days=182)).date()
so = so[so["trandate"] >= six_months_ago]

# Adjust the quantitative columns
adj_list = ["quantity", "netamount","costestimate"]
for col in adj_list:
    so[col] = so[col]*-1

# Filter only the 
so_status = pd.read_csv("so_status.csv")
so = so.merge(right=so_status, on = "status", how = "left")

# Exclude Closed and Cancelled -> Demand doesn't exist
valid_so = so[~so["status"].isin(["H", "C"])]

In [9]:
# individual sales aggregate history
item_sales_agg = valid_so.groupby(by = "item", as_index=False).agg(
    total_amount = ("netamount", "sum"),
    total_quantity = ("quantity", "sum"),
    total_costestimate = ("costestimate", "sum"),
    so_count = ("tranid", "nunique"),
    sales_days = ("trandate", "nunique")
)

item_sales_agg = (item_sales_agg.sort_values(by= "total_amount", ascending=False)
                .assign(cum_pct = lambda x: x["total_amount"].cumsum()/ x["total_amount"].sum())
                .reset_index(drop = True))
threshold = 0.8
x_val = item_sales_agg["cum_pct"].ge(threshold).idxmax()

top_revenue = item_sales_agg.iloc[0:x_val, :]

In [33]:
top_revenue

,item,total_amount,total_quantity,total_costestimate,so_count,sales_days,cum_pct
0,51229,132576.33,470.0,107204.15,204,94,0.022784
1,48720,92762.59,654.0,79731.73,157,88,0.038725
2,4386,89351.47,1745.0,56792.70,182,84,0.054081
3,4594,72459.02,4392.0,46072.07,267,109,0.066533
4,48637,72035.28,486.0,65354.85,121,75,0.078913
...,...,...,...,...,...,...,...
814,55482,1199.96,4.0,0.00,1,1,0.799068
815,11538,1194.70,26.0,819.38,14,13,0.799273
816,22750,1191.89,12.0,882.33,7,6,0.799478
817,54901,1189.93,7.0,754.88,3,3,0.799683


In [15]:
top_revenue_last = pd.read_csv(r"C:\Users\Tzuying\Project\Purchasing\output_csv_files\top_revenue_from_2025-01-01_to_2026-08-10.csv")


In [25]:
A = set(top_revenue["item"])
B = set(top_revenue_last["item_id"])

len(A - B)
len(B - A)

len(A & B)

648

In [27]:
len(B)

1018

In [28]:
items = client.query(query="""Select * from `clean-pilot-456915-t0.NetSuite.item` """).to_dataframe()

c:\Users\Tzuying\Project\BigQuery Pipeline\.venv\Lib\site-packages\google\cloud\bigquery\table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


In [30]:
items[items['id'] == 51229]

,id,fullname,itemtype,upccode,mpn,displayname,vendorname,purchaseunit,parent,class,...,manufacturer,averagecost,cost,lastpurchaseprice,costingmethoddisplay,autoleadtime,autoreorderpoint,autopreferredstocklevel,lastmodifieddate,_loaded_at
18983,51229,3M: Attest Super Rapid Readout Biological Indi...,InvtPart,3M.1492V,1492V,09-002-1492V,NaN,1,<NA>,67,...,3M (Solventum),NaN,238.53,238.53,Average,False,False,False,2026-09-25,2026-09-25 11:19:40.732561+00:00


In [31]:
items[items["averagecost"].isna()]

,id,fullname,itemtype,upccode,mpn,displayname,vendorname,purchaseunit,parent,class,...,manufacturer,averagecost,cost,lastpurchaseprice,costingmethoddisplay,autoleadtime,autoreorderpoint,autopreferredstocklevel,lastmodifieddate,_loaded_at
5,947,Sable: Screw-in Contra Angle 4:1- 1600212,InvtPart,Sa.1600212,1600212,07-458-1600212,NaN,1,<NA>,99,...,Sable Industries,NaN,123.06,0.00,Average,False,False,False,2026-08-17,2026-09-25 11:19:40.732561+00:00
6,948,Discontinued Sable: Access Pro KL - Mini Kavo...,InvtPart,SAI.2000019,2000019,07-458-2000019,NaN,1,<NA>,99,...,Sable Industries,NaN,613.52,515.36,Average,True,True,True,2026-08-14,2026-09-25 11:19:40.732561+00:00
8,950,Discontinued Sable: Non Optic High Speed Rota...,InvtPart,Sa.2000004,2000004,07-458-2000004,NaN,1,<NA>,99,...,Sable Industries,NaN,308.18,290.74,Average,False,False,False,2026-08-14,2026-09-25 11:19:40.732561+00:00
13,955,American Express: Gift Card $125,InvtPart,AME.GIFTAE-125,GIFTAE-125,08-035-GIFTAE-125,NaN,1,<NA>,110,...,American Express,NaN,125.00,127.25,Average,True,True,True,2026-08-14,2026-09-25 11:19:40.732561+00:00
14,956,American Express: Gift Card $150,InvtPart,AME.GIFTAE-150,GIFTAE-150,08-035-GIFTAE-150,NaN,1,<NA>,110,...,American Express,NaN,150.00,152.25,Average,True,True,True,2026-08-14,2026-09-25 11:19:40.732561+00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
34899,74586,D2: Prophy Cups Elite Latch Type Blue 144/Bx -...,InvtPart,D2.PL027LB,PL-027L-B,16-139-PL-027L-B,NaN,1,<NA>,83,...,D2,NaN,NaN,NaN,Average,False,False,False,2026-09-24,2026-09-25 11:19:40.732561+00:00
34900,74587,D2: Prophy Cups Elite Screw Type Blue 144/Box ...,InvtPart,D2.PL027SB,PL-027S-B,16-139-PL-027S-B,NaN,1,<NA>,83,...,D2,NaN,NaN,NaN,Average,False,False,False,2026-09-24,2026-09-25 11:19:40.732561+00:00
34901,74588,HSB - Alginate Dustless Fast Set 1Lb Pouch Typ...,InvtPart,Hs.102110,102110,10-223-102110,NaN,1,<NA>,51,...,House Brand Dentistry,NaN,NaN,NaN,Average,False,False,False,2026-09-25,2026-09-25 11:19:40.732561+00:00
34902,74589,Citagenix: Wound Dressing NeoPlug 10mm x 20mm ...,InvtPart,Ci.NEOPLUG,NEOPLUG,13-116-NEOPLUG,NaN,1,<NA>,14,...,Citagenix,NaN,NaN,NaN,Average,True,True,True,2026-09-25,2026-09-25 11:19:40.732561+00:00
